# Hotel Booking Analytics & Cancellation Prediction---**Author:** Piyush Raikwar  **Program:** B.Tech – Artificial Intelligence & Data Science  **Dataset:** Hotel Booking Demand (hotel_bookings.csv)  **Goal:** Analyse hotel booking patterns and build a machine learning model to predict booking cancellations.

## 1. Problem Statement & Objectives### Business ProblemHotels lose significant revenue to last-minute cancellations.  Understanding *why* bookings get cancelled enables hotels to:- Adjust overbooking strategies- Target at-risk customers with retention offers- Improve revenue forecasting### Objectives1. Perform thorough exploratory data analysis (EDA) on hotel booking data2. Answer 24 specific business questions covering booking, customer, pricing, and cancellation analytics3. Build and compare multiple machine learning models to predict whether a booking will be cancelled4. Deploy findings in an interactive Streamlit dashboard

## 2. Import Libraries

In [ ]:
import sysfrom pathlib import Pathimport warningswarnings.filterwarnings('ignore')import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import (    accuracy_score, precision_score, recall_score,    f1_score, roc_auc_score, confusion_matrix,    ConfusionMatrixDisplay, roc_curve, classification_report)import joblib# Add project root so src modules are importablesys.path.insert(0, str(Path('..').resolve()))from src.data_cleaning import load_raw_data, clean_data, save_cleaned_datafrom src.feature_engineering import engineer_featuresfrom src.preprocessing import build_preprocessor, get_X_y, NUMERICAL_FEATURES, CATEGORICAL_FEATURESfrom src.model_training import (    split_data, build_model_pipelines, evaluate_model,    get_feature_importance, RANDOM_STATE)# Plot settingsplt.rcParams.update({    'figure.figsize': (10, 5),    'axes.titlesize': 13,    'axes.labelsize': 11,    'xtick.labelsize': 9,    'ytick.labelsize': 9,    'figure.dpi': 100,})sns.set_style('whitegrid')PALETTE = sns.color_palette('Set2')print("Libraries loaded successfully.")

## 3. Data Loading

In [ ]:
raw_df = load_raw_data('../data/raw/hotel_bookings.csv')print(f"Dataset shape: {raw_df.shape}")print(f"Rows: {raw_df.shape[0]:,}  |  Columns: {raw_df.shape[1]}")raw_df.head()

## 4. Data Understanding### Dataset OverviewThe **Hotel Booking Demand** dataset contains booking records for two hotel types:- **Resort Hotel** – leisure-oriented property- **City Hotel** – urban, business-oriented propertyThe data spans **July 2015 – August 2017**.

In [ ]:
print("Column names and data types:")print(raw_df.dtypes)

In [ ]:
print("Statistical summary of numerical columns:")raw_df.describe().T.round(2)

## 5. Data Quality Analysis### 5.1 Missing Values

In [ ]:
missing = raw_df.isnull().sum()missing_pct = (missing / len(raw_df) * 100).round(2)missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)print("Columns with missing values:")print(missing_df)

### 5.2 Duplicate Rows

In [ ]:
n_dupes = raw_df.duplicated().sum()print(f"Duplicate rows: {n_dupes:,}  ({n_dupes/len(raw_df)*100:.1f}% of total)")

### 5.3 Target Variable Distribution

In [ ]:
counts = raw_df['is_canceled'].value_counts()pct = (counts / len(raw_df) * 100).round(1)print("is_canceled distribution:")print(pd.DataFrame({'Count': counts, 'Percentage': pct}))fig, axes = plt.subplots(1, 2, figsize=(10, 4))axes[0].bar(['Not Cancelled\n(0)', 'Cancelled\n(1)'], counts.values,            color=[PALETTE[1], PALETTE[3]])axes[0].set_title('Booking Cancellation Distribution')axes[0].set_ylabel('Number of Bookings')for i, v in enumerate(counts.values):    axes[0].text(i, v + 200, f'{v:,}\n({pct.values[i]}%)', ha='center')axes[1].pie(counts.values, labels=['Not Cancelled', 'Cancelled'],            autopct='%1.1f%%', colors=[PALETTE[1], PALETTE[3]],            startangle=90, textprops={'fontsize': 10})axes[1].set_title('Cancellation Rate')plt.tight_layout()plt.savefig('../images/01_cancellation_distribution.png', bbox_inches='tight')plt.show()print("\nInsight: Approximately 37% of bookings in the raw dataset are cancelled - a significant rate.")

### 5.4 Suspicious / Invalid Values

In [ ]:
print("Suspicious values in raw dataset:")print(f"  Adults == 0: {(raw_df['adults']==0).sum()}")print(f"  ADR < 0:     {(raw_df['adr']<0).sum()}")print(f"  ADR > 5000:  {(raw_df['adr']>5000).sum()}")zero_stays = ((raw_df['stays_in_weekend_nights']==0) & (raw_df['stays_in_week_nights']==0))print(f"  Zero total nights: {zero_stays.sum()}")print(f"  Meal 'Undefined':  {(raw_df['meal']=='Undefined').sum()}")

## 6. Data Cleaning### Cleaning Strategy| Issue | Method | Justification ||-------|--------|---------------|| 31,994 duplicate rows | Drop duplicates | Identical records are data entry errors || `children` 4 NaN | Fill with 0 | Most likely no children; numeric context || `country` 488 NaN | Fill with 'Unknown' | Preserves records without location info || `agent` 16,340 NaN | Fill with 0 | NaN means no agent was used || `company` 112,593 NaN | Fill with 0 | NaN means no corporate company || ADR < 0 (1 record) | Median-fill | Physically impossible; likely data entry error || ADR > 5000 (1 record) | Cap at 5000 | Extreme outlier; likely data error || Zero-guest records | Remove | A booking with 0 people is invalid || Meal = 'Undefined' | Map to 'SC' | Domain equivalent: no meal plan |

In [ ]:
clean_df = clean_data(raw_df, verbose=True)save_cleaned_data(clean_df, '../data/processed/cleaned_hotel_bookings.csv')print(f"\nCleaning summary:")print(f"  Raw rows: {len(raw_df):,}")print(f"  Clean rows: {len(clean_df):,}")print(f"  Removed: {len(raw_df) - len(clean_df):,} rows ({(len(raw_df)-len(clean_df))/len(raw_df)*100:.1f}%)")

## 7. Exploratory Data Analysis### 7.1 Hotel Type Distribution

In [ ]:
hotel_counts = clean_df['hotel'].value_counts()fig, ax = plt.subplots(figsize=(7, 4))ax.bar(hotel_counts.index, hotel_counts.values, color=[PALETTE[0], PALETTE[2]])ax.set_title('Number of Bookings by Hotel Type')ax.set_ylabel('Number of Bookings')for i, v in enumerate(hotel_counts.values):    ax.text(i, v + 200, f'{v:,}', ha='center')plt.tight_layout()plt.savefig('../images/02_hotel_type_distribution.png', bbox_inches='tight')plt.show()print(f"City Hotel receives {hotel_counts['City Hotel']/hotel_counts.sum()*100:.1f}% of bookings.")print(f"Resort Hotel receives {hotel_counts['Resort Hotel']/hotel_counts.sum()*100:.1f}% of bookings.")

### 7.2 Monthly Booking Demand Trend

In [ ]:
month_order = ['January','February','March','April','May','June',               'July','August','September','October','November','December']monthly = clean_df.groupby('arrival_date_month').size().reindex(month_order)fig, ax = plt.subplots(figsize=(12, 5))ax.plot(month_order, monthly.values, marker='o', linewidth=2, color=PALETTE[0])ax.fill_between(month_order, monthly.values, alpha=0.2, color=PALETTE[0])ax.set_title('Monthly Booking Demand')ax.set_ylabel('Number of Bookings')ax.set_xlabel('Month')plt.xticks(rotation=30)plt.tight_layout()plt.savefig('../images/03_monthly_demand.png', bbox_inches='tight')plt.show()print(f"Peak month: {monthly.idxmax()} ({monthly.max():,} bookings)")print(f"Lowest month: {monthly.idxmin()} ({monthly.min():,} bookings)")

### 7.3 Monthly Cancellation Rate

In [ ]:
monthly_cancel = clean_df.groupby('arrival_date_month')['is_canceled'].mean().reindex(month_order) * 100fig, ax = plt.subplots(figsize=(12, 5))bars = ax.bar(month_order, monthly_cancel.values, color=PALETTE[3], alpha=0.8)ax.set_title('Monthly Cancellation Rate (%)')ax.set_ylabel('Cancellation Rate (%)')ax.set_xlabel('Month')ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))plt.xticks(rotation=30)plt.tight_layout()plt.savefig('../images/04_monthly_cancellation_rate.png', bbox_inches='tight')plt.show()print(f"Highest cancellation rate: {monthly_cancel.idxmax()} ({monthly_cancel.max():.1f}%)")print(f"Lowest cancellation rate: {monthly_cancel.idxmin()} ({monthly_cancel.min():.1f}%)")

### 7.4 Lead Time Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))axes[0].hist(clean_df['lead_time'], bins=50, color=PALETTE[0], edgecolor='white')axes[0].set_title('Lead Time Distribution')axes[0].set_xlabel('Lead Time (days)')axes[0].set_ylabel('Count')cancelled = clean_df[clean_df['is_canceled']==1]['lead_time']not_cancelled = clean_df[clean_df['is_canceled']==0]['lead_time']axes[1].hist(not_cancelled, bins=50, alpha=0.6, label='Not Cancelled', color=PALETTE[1])axes[1].hist(cancelled, bins=50, alpha=0.6, label='Cancelled', color=PALETTE[3])axes[1].set_title('Lead Time: Cancelled vs Not Cancelled')axes[1].set_xlabel('Lead Time (days)')axes[1].set_ylabel('Count')axes[1].legend()plt.tight_layout()plt.savefig('../images/05_lead_time_distribution.png', bbox_inches='tight')plt.show()print(f"Average lead time – Not Cancelled: {not_cancelled.mean():.0f} days")print(f"Average lead time – Cancelled:     {cancelled.mean():.0f} days")print("Insight: Cancelled bookings show longer average lead times.")

### 7.5 ADR Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))# ADR by hotel typehotel_adr = clean_df.groupby('hotel')['adr'].mean()axes[0].bar(hotel_adr.index, hotel_adr.values, color=[PALETTE[0], PALETTE[2]])axes[0].set_title('Average Daily Rate by Hotel Type')axes[0].set_ylabel('Average ADR (EUR)')for i, v in enumerate(hotel_adr.values):    axes[0].text(i, v + 0.5, f'{v:.1f}', ha='center')# Monthly ADRmonthly_adr = clean_df.groupby('arrival_date_month')['adr'].mean().reindex(month_order)axes[1].plot(month_order, monthly_adr.values, marker='o', linewidth=2, color=PALETTE[2])axes[1].fill_between(month_order, monthly_adr.values, alpha=0.2, color=PALETTE[2])axes[1].set_title('Average Daily Rate by Month')axes[1].set_ylabel('ADR (EUR)')axes[1].set_xlabel('Month')plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)plt.tight_layout()plt.savefig('../images/06_adr_analysis.png', bbox_inches='tight')plt.show()print(f"Overall average ADR: EUR {clean_df['adr'].mean():.2f}")print(f"City Hotel ADR: EUR {hotel_adr['City Hotel']:.2f}")print(f"Resort Hotel ADR: EUR {hotel_adr['Resort Hotel']:.2f}")

### 7.6 Market Segment Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))seg_counts = clean_df['market_segment'].value_counts()axes[0].barh(seg_counts.index, seg_counts.values, color=PALETTE)axes[0].set_title('Bookings by Market Segment')axes[0].set_xlabel('Number of Bookings')seg_cancel = clean_df.groupby('market_segment')['is_canceled'].mean().sort_values(ascending=False) * 100axes[1].barh(seg_cancel.index, seg_cancel.values, color=PALETTE[3])axes[1].set_title('Cancellation Rate by Market Segment')axes[1].set_xlabel('Cancellation Rate (%)')axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))plt.tight_layout()plt.savefig('../images/07_market_segment.png', bbox_inches='tight')plt.show()print("Market Segment cancellation rates:")print(seg_cancel.round(1))

### 7.7 Deposit Type vs Cancellation

In [ ]:
deposit_cancel = clean_df.groupby('deposit_type')['is_canceled'].mean().sort_values(ascending=False) * 100fig, ax = plt.subplots(figsize=(8, 4))ax.bar(deposit_cancel.index, deposit_cancel.values,       color=[PALETTE[3], PALETTE[1], PALETTE[0]])ax.set_title('Cancellation Rate by Deposit Type')ax.set_ylabel('Cancellation Rate (%)')ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))for i, v in enumerate(deposit_cancel.values):    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center')plt.tight_layout()plt.savefig('../images/08_deposit_type_cancellation.png', bbox_inches='tight')plt.show()print("Deposit type cancellation rates:")print(deposit_cancel.round(1))print("\nInsight: Non-refundable deposits show the highest cancellation rate - possibly")print("because guests cancel even knowing they'll lose their deposit.")

### 7.8 Customer Type Distribution & Cancellation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))cust_counts = clean_df['customer_type'].value_counts()axes[0].pie(cust_counts.values, labels=cust_counts.index,            autopct='%1.1f%%', colors=PALETTE, startangle=90)axes[0].set_title('Customer Type Distribution')cust_cancel = clean_df.groupby('customer_type')['is_canceled'].mean().sort_values(ascending=False) * 100axes[1].bar(cust_cancel.index, cust_cancel.values, color=PALETTE)axes[1].set_title('Cancellation Rate by Customer Type')axes[1].set_ylabel('Cancellation Rate (%)')axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))plt.tight_layout()plt.savefig('../images/09_customer_type.png', bbox_inches='tight')plt.show()print("Customer type cancellation rates:")print(cust_cancel.round(1))

### 7.9 Repeat Guests vs Cancellation

In [ ]:
repeat_cancel = clean_df.groupby('is_repeated_guest')['is_canceled'].mean() * 100labels = ['New Guest', 'Repeat Guest']fig, ax = plt.subplots(figsize=(7, 4))ax.bar(labels, repeat_cancel.values, color=[PALETTE[3], PALETTE[1]])ax.set_title('Cancellation Rate: New vs Repeat Guests')ax.set_ylabel('Cancellation Rate (%)')ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))for i, v in enumerate(repeat_cancel.values):    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center')plt.tight_layout()plt.savefig('../images/10_repeat_guest_cancellation.png', bbox_inches='tight')plt.show()pct_repeat = clean_df['is_repeated_guest'].mean() * 100print(f"Repeat guests: {pct_repeat:.1f}% of all bookings")print(f"New guest cancellation rate:    {repeat_cancel[0]:.1f}%")print(f"Repeat guest cancellation rate: {repeat_cancel[1]:.1f}%")print("Insight: Repeat guests cancel at a much lower rate than new guests.")

### 7.10 Correlation Heatmap

In [ ]:
num_cols = ['lead_time','stays_in_weekend_nights','stays_in_week_nights',            'adults','children','babies','is_repeated_guest','previous_cancellations',            'previous_bookings_not_canceled','booking_changes','days_in_waiting_list',            'adr','required_car_parking_spaces','total_of_special_requests','is_canceled']corr = clean_df[num_cols].corr()fig, ax = plt.subplots(figsize=(13, 10))mask = np.triu(np.ones_like(corr, dtype=bool))sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)ax.set_title('Correlation Heatmap of Numerical Features')plt.tight_layout()plt.savefig('../images/11_correlation_heatmap.png', bbox_inches='tight')plt.show()print("Key correlations with is_canceled:")print(corr['is_canceled'].drop('is_canceled').sort_values(key=abs, ascending=False).round(3))

### 7.11 Top Countries by Bookings

In [ ]:
top_countries = clean_df[clean_df['country'] != 'Unknown']['country'].value_counts().head(10)fig, ax = plt.subplots(figsize=(10, 5))ax.barh(top_countries.index[::-1], top_countries.values[::-1], color=PALETTE[0])ax.set_title('Top 10 Countries by Number of Bookings')ax.set_xlabel('Number of Bookings')plt.tight_layout()plt.savefig('../images/12_top_countries.png', bbox_inches='tight')plt.show()print("Top 5 countries:")print(top_countries.head())

### 7.12 Length of Stay Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))stay_data = clean_df.copy()stay_data['total_nights'] = stay_data['stays_in_weekend_nights'] + stay_data['stays_in_week_nights']stay_data = stay_data[stay_data['total_nights'] <= 20]axes[0].hist(stay_data['total_nights'], bins=20, color=PALETTE[0], edgecolor='white')axes[0].set_title('Total Length of Stay Distribution')axes[0].set_xlabel('Total Nights')axes[0].set_ylabel('Count')stay_cancel = stay_data.groupby('total_nights')['is_canceled'].mean() * 100axes[1].plot(stay_cancel.index, stay_cancel.values, color=PALETTE[3], marker='o')axes[1].set_title('Cancellation Rate by Length of Stay')axes[1].set_xlabel('Total Nights')axes[1].set_ylabel('Cancellation Rate (%)')plt.tight_layout()plt.savefig('../images/13_length_of_stay.png', bbox_inches='tight')plt.show()avg_stay = clean_df['stays_in_weekend_nights'].mean() + clean_df['stays_in_week_nights'].mean()print(f"Average total length of stay: {avg_stay:.1f} nights")print(f"Average weekend nights: {clean_df['stays_in_weekend_nights'].mean():.2f}")print(f"Average weekday nights: {clean_df['stays_in_week_nights'].mean():.2f}")

## 8. Key Business Insights### Summary of Analytical Findings| # | Question | Finding ||---|----------|---------|| 1 | Total bookings (cleaned) | ~87,230 || 2 | Cancellation rate | ~27.5% || 3 | Most bookings by hotel | City Hotel (~64%) || 4 | Peak demand month | August || 5 | Lowest demand month | January || 6 | Average lead time | ~80 days || 7 | Average stay | ~3.4 nights || 8 | Repeat guest share | ~3.2% || 9 | Most common segment | Online TA || 10 | Non-refundable deposit cancellation | Very high (~99%) || 11 | Deposit type biggest predictor | Non-refundable = high cancellation || 12 | Lead time relationship | Longer lead time → higher cancellation rate || 13 | Special requests relationship | More requests → lower cancellation |> **Note:** All findings are observational/associative. Causal inference requires controlled experiments.

## 9. Feature EngineeringNew features derived from existing columns:| Feature | Description ||---------|-------------|| `total_stay_nights` | weekend_nights + week_nights || `total_guests` | adults + children + babies || `is_family` | 1 if children > 0 or babies > 0 || `is_weekend_booking` | 1 if weekend nights > 0 || `has_company` | 1 if corporate company associated || `has_agent` | 1 if travel agent used || `cancellation_history_rate` | Previous cancellations / total previous bookings || `room_type_match` | 1 if reserved == assigned room type || `lead_time_bin` | Categorical bucket of lead time || `adr_per_night` | ADR with zero-night guard |

In [ ]:
fe_df = engineer_features(clean_df, verbose=True)print(f"\nFeature engineered dataset shape: {fe_df.shape}")print("New columns added:")new_cols = [c for c in fe_df.columns if c not in clean_df.columns]print(new_cols)

## 10. Data Leakage Prevention### Excluded ColumnsThe following columns are **excluded** from all ML features because they are only known *after* the booking outcome is determined:| Column | Reason for Exclusion ||--------|----------------------|| `reservation_status` | Directly encodes the cancellation outcome ('Canceled', 'Check-Out', 'No-Show') || `reservation_status_date` | Post-outcome date – unavailable at prediction time || `is_canceled` | The target itself |### Additional Exclusions (low predictive value or data administration)| Column | Reason ||--------|--------|| `arrival_date_month` (string) | Replaced by `arrival_month_num` (numeric) || `arrival_date_week_number` | Too granular; year+month sufficient || `arrival_date_day_of_month` | Too granular || `arrival_date_year` | Dataset spans 2015-2017; year is not generalisable |All remaining columns are retained for their predictive value.

## 11. Machine Learning Preprocessing

In [ ]:
X, y = get_X_y(fe_df)print(f"Feature matrix shape: {X.shape}")print(f"Target vector shape:  {y.shape}")print(f"\nNumerical features ({len(NUMERICAL_FEATURES)}):")print(NUMERICAL_FEATURES)print(f"\nCategorical features ({len(CATEGORICAL_FEATURES)}):")print(CATEGORICAL_FEATURES)print(f"\nClass distribution:")print(y.value_counts(normalize=True).round(3))

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)print(f"Training set:  {X_train.shape[0]:,} rows")print(f"Test set:      {X_test.shape[0]:,} rows")print(f"Cancellation rate – train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")

## 12. Model Training### Models Compared1. **Logistic Regression** – Linear baseline; interpretable coefficients2. **Decision Tree** – Non-linear; easy to interpret; prone to overfitting3. **Random Forest** – Ensemble of trees; robust; handles non-linearity4. **Gradient Boosting** – Sequential boosting; typically strong performerAll models use the same preprocessor pipeline (median imputation + standard scaling for numericals; constant imputation + one-hot encoding for categoricals).

In [ ]:
preprocessor = build_preprocessor()pipelines = build_model_pipelines(preprocessor)results = {}for name, pipeline in pipelines.items():    print(f"Training {name}...", end=' ')    pipeline.fit(X_train, y_train)    metrics = evaluate_model(pipeline, X_test, y_test)    results[name] = metrics    print("done")print("\nAll models trained.")

## 13. Model Evaluation

In [ ]:
results_df = pd.DataFrame(results).T.round(4)results_df.columns = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']results_df = results_df.sort_values('ROC-AUC', ascending=False)print("\n=== MODEL COMPARISON TABLE ===")print(results_df.to_string())

In [ ]:
# --- Confusion Matrices ---fig, axes = plt.subplots(2, 2, figsize=(12, 10))axes = axes.flatten()for idx, (name, pipeline) in enumerate(pipelines.items()):    y_pred = pipeline.predict(X_test)    cm = confusion_matrix(y_test, y_pred)    disp = ConfusionMatrixDisplay(cm, display_labels=['Not Cancelled', 'Cancelled'])    disp.plot(ax=axes[idx], colorbar=False, cmap='Blues')    axes[idx].set_title(f'{name}')plt.suptitle('Confusion Matrices – All Models', fontsize=14, y=1.01)plt.tight_layout()plt.savefig('../images/14_confusion_matrices.png', bbox_inches='tight')plt.show()

In [ ]:
# --- ROC Curves ---fig, ax = plt.subplots(figsize=(9, 7))for name, pipeline in pipelines.items():    y_prob = pipeline.predict_proba(X_test)[:, 1]    fpr, tpr, _ = roc_curve(y_test, y_prob)    auc = roc_auc_score(y_test, y_prob)    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', linewidth=2)ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')ax.set_title('ROC Curves – All Models')ax.set_xlabel('False Positive Rate')ax.set_ylabel('True Positive Rate')ax.legend(loc='lower right')plt.tight_layout()plt.savefig('../images/15_roc_curves.png', bbox_inches='tight')plt.show()

In [ ]:
# --- Metric Comparison Bar Chart ---metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']x = np.arange(len(metrics))width = 0.2fig, ax = plt.subplots(figsize=(13, 6))for i, (name, row) in enumerate(results_df.iterrows()):    ax.bar(x + i * width, [row[m] for m in metrics], width, label=name, alpha=0.8)ax.set_xticks(x + width * 1.5)ax.set_xticklabels(metrics)ax.set_ylabel('Score')ax.set_ylim(0, 1.05)ax.set_title('Model Metric Comparison')ax.legend(loc='lower right')plt.tight_layout()plt.savefig('../images/16_model_comparison.png', bbox_inches='tight')plt.show()

## 14. Best Model & Feature Importance### Best Model: Gradient BoostingSelected based on highest **ROC-AUC** score, which measures the model's ability to discriminate between cancelled and non-cancelled bookings across all classification thresholds.**Why ROC-AUC is the primary metric:**- The dataset has moderate class imbalance (~72.5% not cancelled vs 27.5% cancelled)- ROC-AUC is threshold-independent, giving a comprehensive performance view- Hotels care about both avoiding false positives (flagging legitimate guests) and false negatives (missing cancellations)

In [ ]:
best_name = results_df.index[0]best_pipeline = pipelines[best_name]print(f"Best model: {best_name}")print(f"ROC-AUC: {results[best_name]['ROC-AUC']:.4f}")importance_df = get_feature_importance(best_pipeline, list(X.columns))print("\nTop 20 Feature Importances:")print(importance_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))top_n = importance_df.head(15)ax.barh(top_n['feature'][::-1], top_n['importance'][::-1], color=PALETTE[0])ax.set_title(f'Top 15 Feature Importances – {best_name}')ax.set_xlabel('Importance Score')plt.tight_layout()plt.savefig('../images/17_feature_importance.png', bbox_inches='tight')plt.show()

In [ ]:
# Full classification report for best modely_pred_best = best_pipeline.predict(X_test)print(f"=== Classification Report: {best_name} ===\n")print(classification_report(y_test, y_pred_best, target_names=['Not Cancelled', 'Cancelled']))

## 15. Save Model

In [ ]:
import joblibfrom pathlib import PathPath('../models').mkdir(exist_ok=True)joblib.dump(best_pipeline, '../models/model.pkl')joblib.dump(best_name, '../models/best_model_name.pkl')joblib.dump(list(X.columns), '../models/feature_columns.pkl')print("Model saved to models/model.pkl")print("Feature columns saved to models/feature_columns.pkl")

## 16. Final Conclusions### Project SummaryThis project successfully completed a full data science workflow on the Hotel Booking Demand dataset:**Data Cleaning:**- Removed 31,994 duplicate records- Imputed missing values for children, country, agent, company- Corrected 1 negative ADR, capped 1 extreme ADR value- Removed 166 zero-guest invalid records- Final cleaned dataset: 87,230 records**Key Business Findings:**1. **Cancellation rate is ~27.5%** (post-cleaning) – significant revenue at risk2. **City Hotel** receives significantly more bookings than Resort Hotel3. **August** is peak demand; **January** is lowest4. **Non-refundable deposits** show a very high cancellation rate (~99%) – possibly because guests commit to book knowing they cannot cancel, but later cancel anyway5. **Longer lead times** are associated with higher cancellation rates6. **Online TA** is the dominant market segment7. **Repeat guests** cancel at a far lower rate (~0.6%) than new guests (~28%)8. **More special requests** are associated with lower cancellation probability**Machine Learning Results:**| Model | Accuracy | Precision | Recall | F1 | ROC-AUC ||-------|----------|-----------|--------|----|---------|| Gradient Boosting | 0.8178 | 0.7365 | 0.5267 | 0.6141 | 0.8623 || Random Forest | 0.7484 | 0.5278 | 0.8140 | 0.6404 | 0.8577 || Decision Tree | 0.7076 | 0.4822 | 0.8444 | 0.6138 | 0.8387 || Logistic Regression | 0.7207 | 0.4954 | 0.7876 | 0.6082 | 0.8341 |**Best model: Gradient Boosting** – ROC-AUC = 0.8623The most influential features are: deposit_type_Non Refund, lead_time, arrival_month_num, previous_cancellations, and total_of_special_requests.### Limitations- The dataset spans only 2015–2017; temporal drift may affect performance- Geographic coverage is biased toward Portuguese and European guests- The model is trained on historical data; real-time behavioural shifts would require periodic retraining### Future Improvements- Add SHAP explainability for per-prediction interpretations- Incorporate hyperparameter tuning (GridSearchCV/RandomizedSearchCV)- Explore neural network approaches for comparison- Build a real-time API endpoint using FastAPI